# 05 — Estudo de caso: Rio Claro (Nível 2)

**Objetivo:** aprofundar o retrato de Rio Claro dentro do panorama estadual. O
Nível 1 (notebook 04) testa a associação em escala estadual; aqui Rio Claro dá
profundidade, com dados socioeconômicos que não existem para os outros
municípios (CadÚnico, obtido junto à Prefeitura).

**Entradas:** `dataset_municipios_sp_bruto.csv` e `dataset_consolidado_sp.csv`
(notebook 03), mais os dados de Rio Claro em `data/external/` — ver
`FONTES_RIO_CLARO.md` para a citação de cada fonte.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

mun = pd.read_csv(config.DATA_PROCESSED / "dataset_municipios_sp_bruto.csv")
painel = pd.read_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv")

rio_claro = mun[mun["municipio"] == config.RIO_CLARO_NOME].iloc[0]
rc_painel = painel[painel["municipio"] == config.RIO_CLARO_NOME].sort_values("ano")

y = "taxa_internacao_100k_domicilios_idosos"
x = "pct_idosos_sozinhos"
rio_claro


## 5.1 Onde Rio Claro cai na distribuição do estado


In [ ]:
for col, rotulo in [(x, "% de idosos que moram sozinhos"),
                    (y, "Taxa de internação (por 100 mil)")]:
    pct = (mun[col] < rio_claro[col]).mean() * 100
    print(rotulo)
    print(f"  Rio Claro: {rio_claro[col]:,.1f}")
    print(f"  Estado   : média {mun[col].mean():,.1f} | mediana {mun[col].median():,.1f}")
    print(f"  Rio Claro está acima de {pct:.0f}% dos municípios de SP")
    print()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (col, rotulo) in zip(axes, [(x, "% de idosos que moram sozinhos"),
                                     (y, "Internações por 100 mil domicílios")]):
    sns.histplot(mun[col], bins=40, ax=ax, color="#4C72B0", alpha=0.7)
    ax.axvline(mun[col].median(), color="grey", linestyle="--", label="Mediana do estado")
    ax.axvline(rio_claro[col], color="red", linewidth=2, label=config.RIO_CLARO_NOME)
    ax.set_xlabel(rotulo)
    ax.set_ylabel("Nº de municípios")
    ax.legend()

fig.suptitle(f"{config.RIO_CLARO_NOME} na distribuição dos 645 municípios de SP")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "rio_claro_vs_estado.png", dpi=150)
plt.show()


⚠️ **Atenção para a redação do artigo:** Rio Claro tem proporção de idosos
morando sozinhos próxima da mediana estadual, mas taxa de internação **bem
abaixo** da maioria dos municípios de SP. Ou seja, Rio Claro **não** é um caso
extremo que ilustre a hipótese — é um município de perfil médio em "morar
sozinho" e favorável em internação.

Isso não invalida o Nível 2, mas muda o papel dele: Rio Claro serve como **caso
com dado socioeconômico rico** (CadÚnico, seção 5.4), não como "município onde o
problema é mais grave". Apresentá-lo como exemplo do problema contrariaria o
próprio dado — e é o tipo de coisa que um parecerista verifica.


## 5.2 Rio Claro × estado, ano a ano

⚠️ 2026 é parcial (até julho) — aparece pontilhado nos dois gráficos.


In [ ]:
estado = painel.groupby("ano")[y].agg(["mean", "median"]).reset_index()
comparativo = estado.merge(
    rc_painel[["ano", y]].rename(columns={y: "rio_claro"}), on="ano", how="left"
)

cheios = comparativo[comparativo["ano"] != config.ANO_SIH_PARCIAL]
ult = comparativo[comparativo["ano"] >= config.ANO_SIH_PARCIAL - 1]

fig, ax = plt.subplots(figsize=(8, 5))
for col, rot, cor in [("median", "Mediana do estado (SP)", "C0"),
                      ("rio_claro", config.RIO_CLARO_NOME, "red")]:
    ax.plot(cheios["ano"], cheios[col], marker="o", label=rot, color=cor)
    ax.plot(ult["ano"], ult[col], linestyle=":", color=cor, alpha=0.6)

ax.annotate(f"{config.ANO_SIH_PARCIAL} parcial\n(até julho)",
            xy=(config.ANO_SIH_PARCIAL, comparativo["rio_claro"].iloc[-1]),
            xytext=(-12, -38), textcoords="offset points", ha="center", fontsize=9, color="grey")
ax.set_xlabel("Ano")
ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
ax.set_title(f"{config.RIO_CLARO_NOME} × mediana do estado de SP")
ax.set_xticks(config.ANOS_SIH)
ax.legend()
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "rio_claro_vs_estado_serie.png", dpi=150)
plt.show()

print(comparativo.round(1).to_string(index=False))


## 5.3 Tendência de Rio Claro

Regressão linear simples sobre os **anos completos** (2022-2025). 2026 fica de
fora do ajuste por estar incompleto — incluí-lo puxaria a reta para baixo e
sugeriria uma queda que não existe.

⚠️ São só 4 pontos: isto é uma **referência ilustrativa**, não previsão robusta.
Declarar essa limitação no artigo.


In [ ]:
serie = rc_painel[rc_painel["ano"] != config.ANO_SIH_PARCIAL].dropna(subset=[y])

coefs = np.polyfit(serie["ano"], serie[y], 1)
anos_linha = np.arange(serie["ano"].min(), config.ANO_SIH_PARCIAL + 1)
tendencia = np.polyval(coefs, anos_linha)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(serie["ano"], serie[y], label="Observado (anos completos)", zorder=3)
parcial = rc_painel[rc_painel["ano"] == config.ANO_SIH_PARCIAL]
ax.scatter(parcial["ano"], parcial[y], facecolors="none", edgecolors="grey",
           label=f"{config.ANO_SIH_PARCIAL} (parcial, fora do ajuste)", zorder=3)
ax.plot(anos_linha, tendencia, "--", color="red", label="Tendência linear (2022-2025)")

ax.set_xlabel("Ano")
ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
ax.set_title(f"Tendência da taxa de internação — {config.RIO_CLARO_NOME} (ilustrativa)")
ax.set_xticks(config.ANOS_SIH)
ax.legend()
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "projecao_rio_claro.png", dpi=150)
plt.show()

print(f"Variação média: {coefs[0]:+,.0f} internações por 100 mil ao ano (2022-2025)")


## 5.4 Perfil das internações em Rio Claro, por causa


In [ ]:
causas = list(config.CAUSAS_SIH.keys())
perfil_rc = rio_claro[causas].astype(int).rename(index=config.CAUSAS_SIH_LABELS)
perfil_estado = mun[causas].sum().rename(index=config.CAUSAS_SIH_LABELS)

fig, ax = plt.subplots(figsize=(7, 5))
perfil_rc.sort_values().plot(kind="barh", ax=ax, color="#C44E52")
ax.set_xlabel(f"Total de internações ({config.PERIODO_SIH})")
ax.set_title(f"Perfil das internações em idosos, por capítulo CID-10 — {config.RIO_CLARO_NOME}")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "perfil_causas_rio_claro.png", dpi=150)
plt.show()

comparacao = pd.DataFrame({
    f"{config.RIO_CLARO_NOME} (%)": (perfil_rc / perfil_rc.sum() * 100).round(1),
    "Estado de SP (%)": (perfil_estado / perfil_estado.sum() * 100).round(1),
})
print(comparacao.to_string())


## 5.5 Cadastro Único — perfil socioeconômico dos idosos que moram sozinhos

**Fonte:** Ofício SMDS nº 2235/2026, Prefeitura Municipal de Rio Claro, dados de
Junho/2026 (ver `data/external/FONTES_RIO_CLARO.md` para a citação completa).
Dado agregado por faixa de renda, sem identificação individual.

**Cobertura:** o Cadastro Único registra famílias de baixa renda. Os 8.439
idosos aqui representam ~23% do total de idosos do município (37.038, Censo
2022) — esta seção descreve o perfil dos idosos **em situação de
vulnerabilidade socioeconômica**, não de todos os idosos de Rio Claro.


In [ ]:
cadunico = pd.read_csv(config.DATA_EXTERNAL / "cadunico_rio_claro.csv")
cadunico["pct_unipessoal"] = (cadunico["idosos_familias_unipessoais"] / cadunico["total_idosos"] * 100).round(1)

total_idosos_cadunico = int(cadunico["total_idosos"].sum())
total_unipessoais_cadunico = int(cadunico["idosos_familias_unipessoais"].sum())
pct_geral_cadunico = total_unipessoais_cadunico / total_idosos_cadunico * 100

print(f"Idosos no CadÚnico de Rio Claro: {total_idosos_cadunico}")
print(f"Em famílias unipessoais (moram sozinhos): {total_unipessoais_cadunico} ({pct_geral_cadunico:.1f}%)")

cadunico


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(cadunico["faixa_renda"], cadunico["pct_unipessoal"], color="#C44E52")
ax.set_ylabel("% de idosos em famílias unipessoais")
ax.set_title(f"{config.RIO_CLARO_NOME} — idosos que moram sozinhos, por faixa de renda\n(CadÚnico, Jun/2026)")
ax.set_ylim(0, 100)
for i, v in enumerate(cadunico["pct_unipessoal"]):
    ax.text(i, v + 2, f"{v}%", ha="center")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "cadunico_pct_unipessoal_por_renda.png", dpi=150)
plt.show()


**Primeira leitura (a aprofundar no texto do artigo):** entre os idosos em
extrema pobreza cadastrados, 75,2% moram sozinhos — a faixa "Baixa Renda" chama
atenção por destoar bastante das vizinhas (10,2%); vale confirmar esse número
com a Prefeitura antes de usar no artigo, pode ser uma particularidade real da
faixa ou uma inconsistência pontual dos dados.

**Comparação com o Censo (notebook 01):** no Censo 2022, 28,5% dos domicílios de
Rio Claro com responsável idoso são unipessoais (todas as faixas de renda). No
CadÚnico (só baixa renda), a proporção geral é bem maior, 46,1% — consistente
com a ideia de que viver sozinho é mais comum entre idosos de baixa renda,
embora as duas fontes meçam populações e períodos diferentes (Censo 2022, todos
os idosos responsáveis por domicílio; CadÚnico Jun/2026, só idosos de famílias
de baixa renda) e não devam ser comparadas como se fossem a mesma métrica —
vale como leitura qualitativa, não como teste estatístico.


## 5.6 Evolução histórica da população idosa em Rio Claro

**Fonte:** mesmo ofício, Tabela 1 (Censos IBGE 1970-2022) — série oficial
divulgada pela Prefeitura, útil para contextualizar o crescimento do segmento
idoso na introdução do artigo.


In [ ]:
historico = pd.read_csv(config.DATA_EXTERNAL / "censo_rio_claro_historico.csv")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(historico["ano"], historico["pessoas_idosas_pct"], marker="o", color="#4C72B0")
ax.set_xlabel("Ano")
ax.set_ylabel("% de idosos na população")
ax.set_title(f"{config.RIO_CLARO_NOME} — evolução da proporção de idosos (1970-2022)")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "evolucao_idosos_rio_claro.png", dpi=150)
plt.show()

historico


## 5.7 Outros indicadores citados pela Prefeitura (ainda não integrados)

Do mesmo ofício, para uso textual/contextual no artigo:
- **BPC Idoso** (65+, LOAS): 2.130 beneficiários em Rio Claro (Julho/2026)
- **SCFV** (Serviço de Convivência e Fortalecimento de Vínculos): 231 pessoas
  idosas atendidas

Se algum desses números ganhar uma seção própria de análise, criamos aqui.
